In [1]:
#
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras.layers import Input, Conv1D, MaxPooling1D, LSTM, Dense, Flatten, Bidirectional, Lambda
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Layer
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import train_test_split
from scipy import signal
import matplotlib.pyplot as plt
from collections import Counter
import time
import os
from datetime import datetime
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint

import os
import numpy as np
from datetime import datetime
from sklearn.metrics import confusion_matrix, precision_score, recall_score, f1_score, matthews_corrcoef
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint

In [2]:
#

class FuzzyLayer(Layer):
    def __init__(self, output_dim, initializer_centers=None, initializer_sigmas=None, **kwargs):
        super(FuzzyLayer, self).__init__(**kwargs)
        self.output_dim = output_dim
        self.initializer_centers = initializer_centers or 'glorot_uniform'
        self.initializer_sigmas = initializer_sigmas or 'ones'

    def build(self, input_shape):
        self.c = self.add_weight(name='c', 
                                 shape=(input_shape[-1], self.output_dim),
                                 initializer=self.initializer_centers,
                                 trainable=True)
        self.a = self.add_weight(name='a', 
                                 shape=(input_shape[-1], self.output_dim),
                                 initializer=self.initializer_sigmas,
                                 trainable=True)
        super(FuzzyLayer, self).build(input_shape)

    def call(self, x):
        aligned_x = tf.repeat(tf.expand_dims(x, axis=-1), self.output_dim, axis=-1)
        aligned_c = tf.expand_dims(self.c, axis=0)  # 廣播到 batch_size
        aligned_a = tf.expand_dims(self.a, axis=0)
        xc = tf.exp(-tf.reduce_sum(tf.square((aligned_x - aligned_c) / (2 * aligned_a)), axis=-2))
        return xc

    def compute_output_shape(self, input_shape):
        return (input_shape[0], self.output_dim)

    def get_config(self):
        config = super(FuzzyLayer, self).get_config()
        config.update({
            'output_dim': self.output_dim,
            'initializer_centers': self.initializer_centers,
            'initializer_sigmas': self.initializer_sigmas,
        })
        return config
class TSKLayer(Layer):
    def __init__(self, output_dim, initializer_a=None, **kwargs):
        if 'input_shape' not in kwargs and 'input_dim' in kwargs:
            kwargs['input_shape'] = (kwargs.pop('input_dim'),)
        if 'name' not in kwargs:
            kwargs['name'] = 'Weights'
        self.output_dim = output_dim
        self.initializer_a = initializer_a
        super().__init__(**kwargs)

    def build(self, input_shape):
        assert isinstance(input_shape, list) and len(input_shape) == 2
        x_shape, psi_shape = input_shape
        self.a = self.add_weight(name='a',
                                 shape=(1 + x_shape[-1], self.output_dim),
                                 initializer=self.initializer_a if self.initializer_a is not None else 'uniform',
                                 trainable=True)
        super().build(input_shape)

    def call(self, x, **kwargs):
        assert isinstance(x, list) and len(x) == 2
        x, psi = x
        b = tf.ones((tf.shape(x)[0], 1), dtype=x.dtype)
        aligned_b = tf.concat([b, x], axis=-1)
        aligned_a = self.a
        w2 = tf.matmul(aligned_b, aligned_a)
        return psi * w2

    def compute_output_shape(self, input_shape):
        x_shape, psi_shape = input_shape
        return tuple(x_shape[:-1]) + (self.output_dim,)

    def get_config(self):
        base_config = super(TSKLayer, self).get_config()
        base_config['output_dim'] = self.output_dim
        return base_config
def load_uci_data(data_dir='./dataset/UCI HAR Dataset/'):
    """載入 UCI-HAR 資料集"""
    TRAIN = 'train/Inertial Signals/'
    TEST = 'test/Inertial Signals/'

    def load_data(paths):
        data = []
        for path in paths:
            data.append(np.loadtxt(path))
        return np.transpose(np.array(data), (1, 2, 0))  # (samples, timesteps, features)

    INPUT_COLUMNS = [
        "body_gyro_x_", "body_gyro_y_", "body_gyro_z_",
        "total_acc_x_", "total_acc_y_", "total_acc_z_"
    ]

    X_TRAIN_PATHS = [data_dir + TRAIN + col + 'train.txt' for col in INPUT_COLUMNS]
    X_TEST_PATHS = [data_dir + TEST + col + 'test.txt' for col in INPUT_COLUMNS]

    X_train = load_data(X_TRAIN_PATHS)
    X_test = load_data(X_TEST_PATHS)

    # 載入標籤
    y_train = np.loadtxt(data_dir + 'train/y_train.txt')
    y_test = np.loadtxt(data_dir + 'test/y_test.txt')

    # 合併訓練和測試資料
    X = np.vstack((X_train, X_test))
    y = np.concatenate((y_train, y_test))

    return X, y
def load_wisdm_data(phone_accel_path='./dataset/WISDM Dataset/phone/accel/data.csv',
                   phone_gyro_path='./dataset/WISDM Dataset/phone/gyro/data.csv'):
    """載入 WISDM 資料集"""
    # 讀取數據
    phone_accel = pd.read_csv(phone_accel_path)
    phone_gyro = pd.read_csv(phone_gyro_path)

    # 合併 phone 加速度計和陀螺儀數據
    phone_data = pd.merge(phone_accel, phone_gyro, on=['timestamp', 'ID', 'activity'], suffixes=('_accel', '_gyro'))

    # 選擇需要的列
    sensor_columns = ['x_accel', 'y_accel', 'z_accel', 'x_gyro', 'y_gyro', 'z_gyro']

    # 將資料分組並轉換為窗口
    def create_windows_from_df(df, window_size=128, overlap=0.5):
        windows = []
        labels = []

        for activity in df['activity'].unique():
            activity_data = df[df['activity'] == activity]

            for user_id in activity_data['ID'].unique():
                user_data = activity_data[activity_data['ID'] == user_id]

                # 確保時間戳排序
                user_data = user_data.sort_values('timestamp')

                # 提取傳感器數據
                sensor_data = user_data[sensor_columns].values

                # 創建窗口
                step = int(window_size * (1 - overlap))
                for i in range(0, len(sensor_data) - window_size + 1, step):
                    window = sensor_data[i:i + window_size]
                    if len(window) == window_size:  # 確保窗口大小一致
                        windows.append(window)
                        labels.append(activity)

        return np.array(windows), np.array(labels)

    # 創建窗口
    X_wisdm, y_wisdm = create_windows_from_df(phone_data)

    # 重新排列維度以匹配 UCI 格式: (samples, timesteps, features)
    X_wisdm = np.transpose(X_wisdm, (0, 1, 2))

    return X_wisdm, y_wisdm
def align_data():
    """數據對齊與處理"""
    X_uci, y_uci = load_uci_data()
    X_wisdm, y_wisdm = load_wisdm_data()

    # 活動映射
    uci_activities = {
        1: 'walking', 2: 'walking_upstairs', 3: 'walking_downstairs',
        4: 'sitting', 5: 'standing', 6: 'laying'
    }

    wisdm_activities = {
        'A': 'walking', 'B': 'jogging', 'C': 'stairs', 'D': 'sitting',
        'E': 'standing'
    }

    # 共同活動映射
    common_activity_mapping = {
        'walking': 0,
        'sitting': 1,
        'standing': 2
    }

    # 篩選 WISDM 資料
    common_wisdm_indices = [
        i for i, label in enumerate(y_wisdm)
        if wisdm_activities.get(label) in common_activity_mapping
    ]

    X_wisdm_common = X_wisdm[common_wisdm_indices]
    y_wisdm_common = np.array([
        common_activity_mapping[wisdm_activities[y_wisdm[i]]]
        for i in common_wisdm_indices
    ])

    # 篩選 UCI 資料
    common_uci_indices = [
        i for i, label in enumerate(y_uci)
        if uci_activities.get(label) in common_activity_mapping
    ]

    X_uci_common = X_uci[common_uci_indices]
    y_uci_common = np.array([
        common_activity_mapping[uci_activities[y_uci[i]]]
        for i in common_uci_indices
    ])

    # 資料對齊：選取前 6 個特徵（加速度計和陀螺儀）
    X_uci_aligned = X_uci_common[:, :, 0:6]
    X_wisdm_aligned = X_wisdm_common

    # 統一窗口大小為 128
    window_size = 128

    if X_wisdm_aligned.shape[1] != window_size:
        X_wisdm_resampled = []
        for i in range(X_wisdm_aligned.shape[0]):
            window = []
            for j in range(X_wisdm_aligned.shape[2]):
                resampled = signal.resample(X_wisdm_aligned[i, :, j], window_size)
                window.append(resampled)
            X_wisdm_resampled.append(np.array(window).T)

        X_wisdm_aligned = np.array(X_wisdm_resampled)

    # 檢查各類別的分佈情況
    print("UCI 數據集類別分佈:")
    for class_id in range(3):
        count = np.sum(y_uci_common == class_id)
        percentage = count / len(y_uci_common) * 100
        print(f"  類別 {class_id}: {count} 樣本 ({percentage:.2f}%)")

    print("\nWISDM 數據集類別分佈:")
    for class_id in range(3):
        count = np.sum(y_wisdm_common == class_id)
        percentage = count / len(y_wisdm_common) * 100
        print(f"  類別 {class_id}: {count} 樣本 ({percentage:.2f}%)")

    # 返回對齊後的數據
    return X_uci_aligned, y_uci_common, X_wisdm_aligned, y_wisdm_common
def align_and_encode_data():
    """
    對齊、標準化並對數據進行 One-Hot 編碼
    返回處理後的數據和標籤
    """
    from sklearn.preprocessing import OneHotEncoder

    # 調用 align_and_normalize_data 進行初步對齊
    X_uci, y_uci, X_wisdm, y_wisdm = align_data()

    # One-Hot 編碼標籤
    encoder = OneHotEncoder(sparse=False)
    y_uci = encoder.fit_transform(y_uci.reshape(-1, 1))
    y_wisdm = encoder.transform(y_wisdm.reshape(-1, 1))

    return X_uci, y_uci, X_wisdm, y_wisdm
def split_and_setup_domains(source, target):
    """
    分割數據集並設置源域和目標域數據
    """
    # 調用 align_and_encode_data 獲取對齊並編碼後的數據
    X_uci, y_uci, X_wisdm, y_wisdm = align_and_encode_data()

    # 根據源域和目標域進行分割
    if source == 'UCI' and target == 'WISDM':
        # 分割 UCI 數據為源域
        X_train_source, X_test_source, y_train_source, y_test_source = train_test_split(
            X_uci, y_uci, test_size=0.2, random_state=42, stratify=y_uci
        )
        # 分割 WISDM 數據為目標域
        X_train_target, X_test_target, y_train_target, y_test_target = train_test_split(
            X_wisdm, y_wisdm, test_size=0.2, random_state=42, stratify=y_wisdm
        )
    elif source == 'WISDM' and target == 'UCI':
        # 分割 WISDM 數據為源域
        X_train_source, X_test_source, y_train_source, y_test_source = train_test_split(
            X_wisdm, y_wisdm, test_size=0.2, random_state=42, stratify=y_wisdm
        )
        # 分割 UCI 數據為目標域
        X_train_target, X_test_target, y_train_target, y_test_target = train_test_split(
            X_uci, y_uci, test_size=0.2, random_state=42, stratify=y_uci
        )
    else:
        raise ValueError(f"不支持的域設置: source={source}, target={target}")

    print(f"\n源域 ({source}) 訓練數據形狀: {X_train_source.shape}, 標籤形狀: {y_train_source.shape}")
    print(f"源域 ({source}) 測試數據形狀: {X_test_source.shape}, 標籤形狀: {y_test_source.shape}")
    print(f"目標域 ({target}) 訓練數據形狀: {X_train_target.shape}, 標籤形狀: {y_train_target.shape}")
    print(f"目標域 ({target}) 測試數據形狀: {X_test_target.shape}, 標籤形狀: {y_test_target.shape}")

    return {
        'source': {
            'train': {'data': X_train_source, 'labels': y_train_source},
            'test': {'data': X_test_source, 'labels': y_test_source}
        },
        'target': {
            'train': {'data': X_train_target, 'labels': y_train_target},
            'test': {'data': X_test_target, 'labels': y_test_target}
        }
    }
def normalize_data(data_dict, method='domain_specific'):
    """
    對數據進行標準化
    method: 標準化方法 ('global', 'per_sample', 'domain_specific')
    """
    from sklearn.preprocessing import StandardScaler

    # 初始化標準化器
    scaler_source = StandardScaler()
    scaler_target = StandardScaler()

    # 全局標準化
    if method == 'global':
        # 對源域訓練數據進行標準化
        X_train_source = data_dict['source']['train']['data']
        X_train_source_reshaped = X_train_source.reshape(-1, X_train_source.shape[2])
        scaler_source.fit(X_train_source_reshaped)

        # 對所有數據進行標準化
        for domain in ['source', 'target']:
            for split in ['train', 'test']:
                X = data_dict[domain][split]['data']
                X_reshaped = X.reshape(-1, X.shape[2])
                X_scaled = scaler_source.transform(X_reshaped)
                data_dict[domain][split]['data'] = X_scaled.reshape(X.shape)

    # 按樣本標準化
    elif method == 'per_sample':
        for domain in ['source', 'target']:
            for split in ['train', 'test']:
                X = data_dict[domain][split]['data']
                for i in range(len(X)):
                    mean = np.mean(X[i], axis=0)
                    std = np.std(X[i], axis=0)
                    X[i] = (X[i] - mean) / (std + 1e-8)
                data_dict[domain][split]['data'] = X

    # 源域和目標域分別標準化
    elif method == 'domain_specific':
        # 對源域訓練數據進行標準化
        X_train_source = data_dict['source']['train']['data']
        X_train_source_reshaped = X_train_source.reshape(-1, X_train_source.shape[2])
        scaler_source.fit(X_train_source_reshaped)

        # 對目標域訓練數據進行標準化
        X_train_target = data_dict['target']['train']['data']
        X_train_target_reshaped = X_train_target.reshape(-1, X_train_target.shape[2])
        scaler_target.fit(X_train_target_reshaped)

        # 對源域數據進行標準化
        for split in ['train', 'test']:
            X = data_dict['source'][split]['data']
            X_reshaped = X.reshape(-1, X.shape[2])
            X_scaled = scaler_source.transform(X_reshaped)
            data_dict['source'][split]['data'] = X_scaled.reshape(X.shape)

        # 對目標域數據進行標準化
        for split in ['train', 'test']:
            X = data_dict['target'][split]['data']
            X_reshaped = X.reshape(-1, X.shape[2])
            X_scaled = scaler_target.transform(X_reshaped)
            data_dict['target'][split]['data'] = X_scaled.reshape(X.shape)

    else:
        raise ValueError(f"不支持的標準化方法: {method}")

    print(f"\n標準化完成，方法: {method}")
    return data_dict
def train_fuzzy_tsk_mmd_model(source_train_data, source_train_labels, target_train_data,
                             source_val_data=None, source_val_labels=None,
                             target_val_data=None, target_val_labels=None,
                             input_shape=None, num_classes=None, num_rules=3,
                             batch_size=64,epochs=100, learning_rate=0.001,
                             mmd_strategy='progressive', start_mmd_weight=0.05,
                             max_mmd_weight=0.5 ,mmd_increase_rate=0.05,
                             checkpoint_path=None, patience=15):
    """
    訓練具有多核MMD損失的FuzzyTSK模型，並監控目標域的準確率
    """
    import numpy as np
    import tensorflow as tf
    import time
    from tqdm import tqdm  # 用於顯示進度條

    # 如果未提供輸入形狀和類別數，則從數據中推斷
    if input_shape is None:
        input_shape = source_train_data.shape[1:]
    if num_classes is None:
        num_classes = source_train_labels.shape[1]
    
    # 創建模型
    model, feature_extractor = create_bilstm_cnn_fuzzy_tsk_model_with_mmd(
        input_shape=input_shape,
        num_classes=num_classes,
        num_rules=num_rules
    )
    
    # 打印模型架構
    print("\n模型架構:")
    model.summary()
    
    # 編譯模型
    optimizer = tf.keras.optimizers.Adam(learning_rate=learning_rate)
    loss_fn = tf.keras.losses.CategoricalCrossentropy()
    
    # 定義多核MMD損失函數
    def multi_kernel_mmd(source_features, target_features):
        """
        使用多個帶寬的高斯核計算MMD，捕捉不同尺度的分布差異
        """
        # 定義多個核帶寬參數
        sigmas = [0.01, 0.1, 1, 10, 100]
        
        # 計算特徵維度
        n_source = tf.cast(tf.shape(source_features)[0], tf.float32)
        n_target = tf.cast(tf.shape(target_features)[0], tf.float32)
        
        total_mmd = 0.0
        
        for sigma in sigmas:
            # 計算高斯核矩陣
            def gaussian_kernel(x, y, sigma=1.0):
                """
                計算高斯核矩陣 K(x, y) = exp(-||x-y||^2 / (2*sigma^2))
                """
                x_size = tf.shape(x)[0]
                y_size = tf.shape(y)[0]
                dim = tf.shape(x)[1]
                
                # 擴展維度以便進行廣播計算
                x = tf.expand_dims(x, 1)  # shape: (x_size, 1, dim)
                y = tf.expand_dims(y, 0)  # shape: (1, y_size, dim)
                
                # 計算歐氏距離的平方
                dist_squared = tf.reduce_sum(tf.square(x - y), axis=2)  # shape: (x_size, y_size)
                
                # 應用高斯核
                return tf.exp(-dist_squared / (2 * sigma**2))
            
            # 計算三個核矩陣
            K_ss = gaussian_kernel(source_features, source_features, sigma)
            K_tt = gaussian_kernel(target_features, target_features, sigma)
            K_st = gaussian_kernel(source_features, target_features, sigma)
            
            # 計算當前核的MMD值
            mmd = tf.reduce_sum(K_ss) / (n_source * n_source)
            mmd += tf.reduce_sum(K_tt) / (n_target * n_target)
            mmd -= 2 * tf.reduce_sum(K_st) / (n_source * n_target)
            
            # 累加到總MMD值
            total_mmd += mmd
        
        return total_mmd
    
    # 儲存最佳模型的變數
    best_val_target_accuracy = 0.0
    best_mixed_metric = -float('inf')  # 初始化最佳混合指標
    patience_counter = 0
    
    # 訓練歷史記錄
    history = {
        'train_loss': [],
        'classification_loss': [],
        'mmd_loss': [],
        'train_accuracy': [],
        'val_source_accuracy': [],
        'val_target_accuracy': [],
        'mmd_weight': [],
        'mixed_metric': []  # 新增混合指標歷史記錄
    }
    
    # 訓練循環
    mmd_weight = start_mmd_weight
    
    for epoch in range(epochs):
        print(f"\nEpoch {epoch+1}/{epochs}")
        
        # 更新MMD權重
        if mmd_strategy == 'progressive':
            mmd_weight = min(start_mmd_weight + epoch * mmd_increase_rate, max_mmd_weight)
        elif mmd_strategy == 'decay':
            mmd_weight = max(max_mmd_weight - epoch * mmd_increase_rate, start_mmd_weight)
        elif mmd_strategy == 'static':
            mmd_weight = start_mmd_weight
        
        # 訓練一個epoch
        start_time = time.time()
        
        # 準備批次數據
        indices = np.random.permutation(len(source_train_data))
        source_train_data_shuffled = source_train_data[indices]
        source_train_labels_shuffled = source_train_labels[indices]
        
        # 確保目標域數據有足夠的樣本
        if len(target_train_data) < len(source_train_data):
            target_indices = np.random.choice(len(target_train_data), len(source_train_data), replace=True)
            target_train_data_used = target_train_data[target_indices]
        else:
            target_indices = np.random.permutation(len(target_train_data))[:len(source_train_data)]
            target_train_data_used = target_train_data[target_indices]
        
        # 批次訓練
        epoch_loss = 0
        epoch_class_loss = 0
        epoch_mmd_loss = 0
        epoch_accuracy = 0
        num_batches = 0
        
        # 使用 tqdm 顯示進度條
        for i in tqdm(range(0, len(source_train_data), batch_size), desc=f"Epoch {epoch+1}/{epochs}"):
            end = min(i + batch_size, len(source_train_data))
            batch_size_actual = end - i
            
            # 獲取源域批次
            source_batch_data = source_train_data_shuffled[i:end]
            source_batch_labels = source_train_labels_shuffled[i:end]
            
            # 獲取目標域批次
            target_batch_data = target_train_data_used[i:end]
            
            with tf.GradientTape() as tape:
                # 源域前向傳播
                source_pred = model(source_batch_data, training=True)
                source_features = feature_extractor(source_batch_data, training=True)
                
                # 目標域前向傳播
                target_features = feature_extractor(target_batch_data, training=True)
                
                # 計算分類損失
                classification_loss = loss_fn(source_batch_labels, source_pred)
                
                # 計算多核MMD損失
                domain_mmd_loss = multi_kernel_mmd(source_features, target_features)
                
                # 總損失
                total_loss = classification_loss + mmd_weight * domain_mmd_loss
            
            # 計算梯度並更新權重
            gradients = tape.gradient(total_loss, model.trainable_variables)
            optimizer.apply_gradients(zip(gradients, model.trainable_variables))
            
            # 計算準確率
            pred_classes = tf.argmax(source_pred, axis=1)
            true_classes = tf.argmax(source_batch_labels, axis=1)
            batch_accuracy = tf.reduce_mean(tf.cast(tf.equal(pred_classes, true_classes), tf.float32))
            
            # 更新統計信息
            epoch_loss += total_loss.numpy()
            epoch_class_loss += classification_loss.numpy()
            epoch_mmd_loss += domain_mmd_loss.numpy()
            epoch_accuracy += batch_accuracy.numpy()
            num_batches += 1
        
        # 計算平均損失和準確率
        epoch_loss /= num_batches
        epoch_class_loss /= num_batches
        epoch_mmd_loss /= num_batches
        epoch_accuracy /= num_batches
        
        # 驗證
        val_source_accuracy = 0
        val_target_accuracy = 0
        
        if source_val_data is not None and source_val_labels is not None:
            source_val_pred = model.predict(source_val_data)
            source_val_pred_classes = np.argmax(source_val_pred, axis=1)
            source_val_true_classes = np.argmax(source_val_labels, axis=1)
            val_source_accuracy = np.mean(source_val_pred_classes == source_val_true_classes)
        
        if target_val_data is not None and target_val_labels is not None:
            target_val_pred = model.predict(target_val_data)
            target_val_pred_classes = np.argmax(target_val_pred, axis=1)
            target_val_true_classes = np.argmax(target_val_labels, axis=1)
            val_target_accuracy = np.mean(target_val_pred_classes == target_val_true_classes)
        
        # 計算混合指標
        beta = 0.7  # 目標域權重
        alpha = 0.3  # 源域權重
        gamma = 0.1  # MMD 損失權重（負值因為我們希望最小化 MMD）
        
        mixed_metric = beta * val_target_accuracy + alpha * val_source_accuracy - gamma * epoch_mmd_loss
        
        # 儲存最佳模型（基於混合指標）
        if mixed_metric > best_mixed_metric:
            best_mixed_metric = mixed_metric
            best_val_target_accuracy = val_target_accuracy  # 同時更新最佳目標域準確率
            patience_counter = 0
            if checkpoint_path:
                model.save_weights(checkpoint_path)  # 改為儲存權重
                print(f"儲存最佳權重，混合指標 = {best_mixed_metric:.4f}, 目標域驗證準確率 = {val_target_accuracy:.4f}")
        else:
            patience_counter += 1
        
        # 早停
        if patience_counter >= patience:
            print(f"早停於第 {epoch+1} 個 epoch，最佳混合指標 = {best_mixed_metric:.4f}, 最佳目標域驗證準確率 = {best_val_target_accuracy:.4f}")
            break
        
        # 記錄歷史
        history['train_loss'].append(epoch_loss)
        history['classification_loss'].append(epoch_class_loss)
        history['mmd_loss'].append(epoch_mmd_loss)
        history['train_accuracy'].append(epoch_accuracy)
        history['val_source_accuracy'].append(val_source_accuracy)
        history['val_target_accuracy'].append(val_target_accuracy)
        history['mmd_weight'].append(mmd_weight)
        history['mixed_metric'].append(mixed_metric)  # 記錄混合指標
        
        # 打印進度
        elapsed_time = time.time() - start_time
        print(f"Loss: {epoch_loss:.4f}, Class Loss: {epoch_class_loss:.4f}, MMD Loss: {epoch_mmd_loss:.4f}, Acc: {epoch_accuracy:.4f}")
        print(f"Val Source Acc: {val_source_accuracy:.4f}, Val Target Acc: {val_target_accuracy:.4f}")
        print(f"Mixed Metric: {mixed_metric:.4f}, MMD Weight: {mmd_weight:.4f}, Time: {elapsed_time:.2f}s")
    
    # 加載最佳權重
    if checkpoint_path:
        model.load_weights(checkpoint_path)
        print("加載最佳權重完成")
    
    return model, history
def evaluate_model(model, data, labels_onehot, true_labels, domain_name="", save_dir=None):
    """
    評估模型性能，並計算多個評估指標。
    """
    # 預測
    predictions = model.predict(data)
    pred_classes = np.argmax(predictions, axis=1)
    true_classes = np.argmax(labels_onehot, axis=1) if labels_onehot is not None else true_labels
    
    # 計算準確率
    accuracy = np.mean(pred_classes == true_classes)
    
    # 計算混淆矩陣
    cm = confusion_matrix(true_classes, pred_classes)
    
    # 計算其他指標
    precision = precision_score(true_classes, pred_classes, average='weighted')
    recall = recall_score(true_classes, pred_classes, average='weighted')
    f1 = f1_score(true_classes, pred_classes, average='weighted')
    mcc = matthews_corrcoef(true_classes, pred_classes)
    
    # 打印結果
    print(f"{domain_name} 評估結果:")
    print(f"準確率: {accuracy:.4f}")
    print(f"Precision: {precision:.4f}")
    print(f"Recall: {recall:.4f}")
    print(f"F1-score: {f1:.4f}")
    print(f"MCC: {mcc:.4f}")
    print(f"混淆矩陣:\n{cm}")
    
    # 保存混淆矩陣到文件
    if save_dir:
        cm_file = os.path.join(save_dir, f"{domain_name}_confusion_matrix.csv")
        np.savetxt(cm_file, cm, fmt='%d', delimiter=',')
        print(f"混淆矩陣已保存至: {cm_file}")
    
    return accuracy, precision, recall, f1, mcc, cm
def visualize_training_history(history, save_path=None):
    """
    可視化訓練歷史
    """
    plt.figure(figsize=(15, 10))
    
    # 繪製損失
    plt.subplot(2, 2, 1)
    plt.plot(history['train_loss'], label='總損失')
    plt.plot(history['classification_loss'], label='分類損失')
    plt.plot(history['mmd_loss'], label='MMD損失')
    plt.title('訓練損失')
    plt.xlabel('Epoch')
    plt.ylabel('損失')
    plt.legend()
    plt.grid(True)
    
    # 繪製準確率
    plt.subplot(2, 2, 2)
    plt.plot(history['train_accuracy'], label='訓練準確率')
    plt.plot(history['val_source_accuracy'], label='源域驗證準確率')
    plt.plot(history['val_target_accuracy'], label='目標域驗證準確率')
    plt.title('訓練與驗證準確率')
    plt.xlabel('Epoch')
    plt.ylabel('準確率')
    plt.legend()
    plt.grid(True)
    
    plt.tight_layout()
    
    if save_path:
        plt.savefig(save_path)
        plt.close()
    else:
        plt.show()
def plot_training_history(history):
    """
    可視化訓練歷史，包括損失、準確率和混合指標
    """
    epochs = range(1, len(history['train_loss']) + 1)

    # 繪製損失
    plt.figure(figsize=(12, 4))
    plt.subplot(1, 3, 1)
    plt.plot(epochs, history['train_loss'], label='Train Loss')
    plt.plot(epochs, history['classification_loss'], label='Classification Loss')
    plt.plot(epochs, history['mmd_loss'], label='MMD Loss')
    plt.title('Loss')
    plt.xlabel('Epochs')
    plt.ylabel('Loss')
    plt.legend()

    # 繪製準確率
    plt.subplot(1, 3, 2)
    plt.plot(epochs, history['train_accuracy'], label='Train Accuracy')
    plt.plot(epochs, history['val_source_accuracy'], label='Val Source Accuracy')
    plt.plot(epochs, history['val_target_accuracy'], label='Val Target Accuracy')
    plt.title('Accuracy')
    plt.xlabel('Epochs')
    plt.ylabel('Accuracy')
    plt.legend()

    # 繪製混合指標
    plt.subplot(1, 3, 3)
    plt.plot(epochs, history['mixed_metric'], label='Mixed Metric')
    plt.title('Mixed Metric')
    plt.xlabel('Epochs')
    plt.ylabel('Metric')
    plt.legend()

    plt.tight_layout()
    plt.show()

In [3]:
def create_bilstm_cnn_fuzzy_tsk_model_with_mmd(input_shape, num_classes, num_rules=3):
    """
    創建先 BiLSTM 後 CNN 的模型，再接 FuzzyTSK 層，並支持 MMD 損失計算
    """
    # 輸入層
    inputs = Input(shape=input_shape)

    # 特徵提取器 (BiLSTM + CNN)
    x = Bidirectional(LSTM(64, return_sequences=True))(inputs)  # 雙向 LSTM
    x = Bidirectional(LSTM(64, return_sequences=True))(x)  # 雙向 LSTM，返回序列以便後續 CNN 處理
    #x = LSTM(64, return_sequences=True)(inputs)  # 雙向 LSTM
    #x = LSTM(64, return_sequences=True)(x)  # 雙向 LSTM，返回序列以便後續 CNN 處理
    x = Conv1D(128, kernel_size=3, activation='relu', padding='same')(x)
    x = MaxPooling1D(pool_size=2)(x)
    x = Conv1D(128, kernel_size=3, activation='relu', padding='same')(x)
    x = MaxPooling1D(pool_size=2)(x)
    fusion = Flatten()(x)  # 扁平化，作為特徵表示層

    # FuzzyTSK 層
    phi = FuzzyLayer(num_rules)(fusion)  # 模糊化輸出
    tsk_output = TSKLayer(num_rules)([fusion, phi])  # TSK 推理

    # 分類輸出層
    outputs = Dense(num_classes, activation='softmax')(tsk_output)

    # 創建模型
    model = Model(inputs=inputs, outputs=outputs)

    # 特徵提取器 (用於 MMD 計算)
    feature_extractor = Model(inputs=inputs, outputs=fusion)

    return model, feature_extractor

In [4]:
# UCI to WISDM baseline
import numpy as np

def main():
    """
    主函數：執行基線模型（不加 MMD）的遷移學習實驗流程
    """
    print("開始執行基線模型實驗（不加 MMD）")
    
    # 創建結果目錄
    results_dir = f"results_baseline_{datetime.now().strftime('%Y%m%d_%H%M%S')}"
    os.makedirs(results_dir, exist_ok=True)
    
    # 加載並處理數據
    print("加載並處理數據...")
    domains = split_and_setup_domains(source='WISDM', target='UCI')  # 調換源域和目標域

    # 對數據進行標準化
    print("對數據進行標準化...")
    domains = normalize_data(domains, method='domain_specific')

    # 提取訓練、測試數據
    source_train_data = domains['source']['train']['data']
    source_train_labels = domains['source']['train']['labels']
    source_test_data = domains['source']['test']['data']
    source_test_labels = domains['source']['test']['labels']

    target_train_data = domains['target']['train']['data']
    target_train_labels = domains['target']['train']['labels']
    target_test_data = domains['target']['test']['data']
    target_test_labels = domains['target']['test']['labels']

    # 分割訓練數據為訓練集和驗證集
    source_train_data, source_val_data, source_train_labels, source_val_labels = train_test_split(
        source_train_data, source_train_labels, test_size=0.2, random_state=42, stratify=source_train_labels
    )
    target_train_data, target_val_data, target_train_labels, target_val_labels = train_test_split(
        target_train_data, target_train_labels, test_size=0.2, random_state=42, stratify=target_train_labels
    )

    print(f"源域數據集: 訓練集 {len(source_train_data)}筆, 驗證集 {len(source_val_data)}筆, 測試集 {len(source_test_data)}筆")
    print(f"目標域數據集: 訓練集 {len(target_train_data)}筆, 驗證集 {len(target_val_data)}筆, 測試集 {len(target_test_data)}筆")
    
    # 超參數
    best_params = {
        'num_rules': 3,          # 規則數量
        'batch_size': 64,        # 批次大小
        'learning_rate': 0.001,  # 學習率
        'mmd_weight': 0.0
    }

    # 訓練多次並儲存結果
    num_repeats = 3  # 設定重複次數
    all_results = []  # 儲存每次訓練的結果
    
    for i in range(num_repeats):
        print(f"\n開始第 {i+1} 次訓練...")
        
        # 創建專屬於本次訓練的結果目錄
        repeat_dir = os.path.join(results_dir, f"repeat_{i+1}")
        os.makedirs(repeat_dir, exist_ok=True)
        
        # 定義 Early Stopping 和 Model Checkpoint
        early_stopping = EarlyStopping(
            monitor='val_accuracy',  # 基於驗證準確率進行早停
            mode='max',
            patience=15,
            verbose=1
        )

        checkpoint_path = os.path.join(repeat_dir, "2LSTM-CTFNN_baseline_wisdm_to_uci_domain_specific.weights.h5")  # 修改檔案名稱
        model_checkpoint = ModelCheckpoint(
            filepath=checkpoint_path,
            monitor='val_accuracy',  # 基於驗證準確率保存最佳模型
            mode='max',
            save_best_only=True,
            verbose=1
        )
        
        # 訓練模型 - 使用驗證集而非測試集進行驗證
        history = train_fuzzy_tsk_mmd_model(
            source_train_data=source_train_data,
            source_train_labels=source_train_labels,
            target_train_data=target_train_data,
            source_val_data=source_val_data,  # 使用驗證集
            source_val_labels=source_val_labels,  # 使用驗證集
            target_val_data=target_val_data,  # 使用驗證集
            target_val_labels=target_val_labels,  # 使用驗證集
            input_shape=source_train_data.shape[1:],  # 輸入數據形狀
            num_classes=source_train_labels.shape[1],  # 類別數
            num_rules=best_params['num_rules'],
            batch_size=best_params['batch_size'],
            learning_rate=best_params['learning_rate'],
            mmd_strategy='static',
            start_mmd_weight=best_params['mmd_weight'],
            epochs=50,
            checkpoint_path=checkpoint_path,  # 傳入模型保存路徑
            patience=15  # 傳入早停的耐心值
        )
        
        # 載入最佳權重
        print("\n載入最佳模型權重...")
        model, _ = create_bilstm_cnn_fuzzy_tsk_model_with_mmd(
            input_shape=source_train_data.shape[1:],
            num_classes=source_train_labels.shape[1],
            num_rules=best_params['num_rules']
        )
        model.load_weights(checkpoint_path)

        # 評估模型 - 使用測試集進行最終評估
        print("\n評估模型性能...")
        source_acc, source_precision, source_recall, source_f1, source_mcc, cm_source = evaluate_model(
            model, source_test_data, source_test_labels, "源域 (WISDM)"  # 修改描述
        )

        target_acc, target_precision, target_recall, target_f1, target_mcc, cm_target = evaluate_model(
            model, target_test_data, target_test_labels, "目標域 (UCI)"  # 修改描述
        )

        print(f"源域測試集準確率: {source_acc:.4f}")
        print(f"目標域測試集準確率: {target_acc:.4f}")

        # 儲存結果
        all_results.append({
            "repeat": i + 1,
            "source_acc": source_acc,
            "source_precision": source_precision,  # 新增精確率
            "source_recall": source_recall,
            "source_f1": source_f1,
            "source_mcc": source_mcc,
            "target_acc": target_acc,
            "target_precision": target_precision,  # 新增精確率
            "target_recall": target_recall,
            "target_f1": target_f1,
            "target_mcc": target_mcc,
            "cm_source": cm_source,
            "cm_target": cm_target
        })

    # 計算平均值和標準差
    metrics_keys = ["source_acc", "source_precision", "source_recall", "source_f1", "source_mcc", 
                    "target_acc", "target_precision", "target_recall", "target_f1", "target_mcc"]
    summary_stats = {key: {"mean": None, "std": None} for key in metrics_keys}

    for key in metrics_keys:
        values = [result[key] for result in all_results]
        summary_stats[key]["mean"] = np.mean(values)
        summary_stats[key]["std"] = np.std(values)

    # 將所有結果保存到總結文件
    summary_file = os.path.join(results_dir, "summary_results_domain_specific.txt")
    with open(summary_file, "w") as f:
        for result in all_results:
            f.write(f"Repeat {result['repeat']}:\n")
            f.write(f"源域準確率: {result['source_acc']:.4f}, Precision: {result['source_precision']:.4f}, "
                    f"Recall: {result['source_recall']:.4f}, F1-score: {result['source_f1']:.4f}, MCC: {result['source_mcc']:.4f}\n")
            f.write(f"目標域準確率: {result['target_acc']:.4f}, Precision: {result['target_precision']:.4f}, "
                    f"Recall: {result['target_recall']:.4f}, F1-score: {result['target_f1']:.4f}, MCC: {result['target_mcc']:.4f}\n")
            f.write("\n")
        
        f.write("\n平均值與標準差:\n")
        for key in metrics_keys:
            f.write(f"{key}: 平均值 = {summary_stats[key]['mean']:.4f} ± {summary_stats[key]['std']:.4f}\n")

    print("\n實驗完成！所有結果已保存。")
    
if __name__ == "__main__":
    main()

開始執行基線模型實驗（不加 MMD）
加載並處理數據...
UCI 數據集類別分佈:
  類別 0: 1722 樣本 (31.86%)
  類別 1: 1777 樣本 (32.88%)
  類別 2: 1906 樣本 (35.26%)

WISDM 數據集類別分佈:
  類別 0: 2308 樣本 (30.55%)
  類別 1: 2742 樣本 (36.29%)
  類別 2: 2506 樣本 (33.17%)

源域 (WISDM) 訓練數據形狀: (6044, 128, 6), 標籤形狀: (6044, 3)
源域 (WISDM) 測試數據形狀: (1512, 128, 6), 標籤形狀: (1512, 3)
目標域 (UCI) 訓練數據形狀: (4324, 128, 6), 標籤形狀: (4324, 3)
目標域 (UCI) 測試數據形狀: (1081, 128, 6), 標籤形狀: (1081, 3)
對數據進行標準化...

標準化完成，方法: domain_specific
源域數據集: 訓練集 4835筆, 驗證集 1209筆, 測試集 1512筆
目標域數據集: 訓練集 3459筆, 驗證集 865筆, 測試集 1081筆

開始第 1 次訓練...

模型架構:
Model: "model"
__________________________________________________________________________________________________
Layer (type)                    Output Shape         Param #     Connected to                     
input_1 (InputLayer)            [(None, 128, 6)]     0                                            
__________________________________________________________________________________________________
bidirectional (Bidirectional)   (None, 

Epoch 1/50: 100%|██████████████████████████████████████████████████████████████████████| 76/76 [00:25<00:00,  3.03it/s]


儲存最佳權重，混合指標 = 0.4656, 目標域驗證準確率 = 0.4173
Loss: 0.8667, Class Loss: 0.8667, MMD Loss: 0.1715, Acc: 0.5761
Val Source Acc: 0.6352, Val Target Acc: 0.4173
Mixed Metric: 0.4656, MMD Weight: 0.0000, Time: 27.37s

Epoch 2/50


Epoch 2/50: 100%|██████████████████████████████████████████████████████████████████████| 76/76 [00:12<00:00,  6.17it/s]


Loss: 0.7695, Class Loss: 0.7695, MMD Loss: 0.1958, Acc: 0.6348
Val Source Acc: 0.6410, Val Target Acc: 0.3815
Mixed Metric: 0.4398, MMD Weight: 0.0000, Time: 13.03s

Epoch 3/50


...

Loss: 0.0851, Class Loss: 0.0851, MMD Loss: 0.2031, Acc: 0.9953
Val Source Acc: 0.9884, Val Target Acc: 0.6023
Mixed Metric: 0.6978, MMD Weight: 0.0000, Time: 12.97s

Epoch 23/50


Epoch 23/50: 100%|█████████████████████████████████████████████████████████████████████| 76/76 [00:12<00:00,  6.18it/s]


Loss: 0.1004, Class Loss: 0.1004, MMD Loss: 0.2194, Acc: 0.9899
Val Source Acc: 0.9868, Val Target Acc: 0.5723
Mixed Metric: 0.6747, MMD Weight: 0.0000, Time: 12.95s

Epoch 24/50


Epoch 24/50: 100%|█████████████████████████████████████████████████████████████████████| 76/76 [00:12<00:00,  6.13it/s]


早停於第 24 個 epoch，最佳混合指標 = 0.8243, 最佳目標域驗證準確率 = 0.7977
加載最佳權重完成

載入最佳模型權重...

評估模型性能...
 評估結果:
準確率: 0.9643
Precision: 0.9649
Recall: 0.9643
F1-score: 0.9642
MCC: 0.9467
混淆矩陣:
[[461   1   0]
 [  1 535  13]
 [  1  38 462]]
 評估結果:
準確率: 0.7798
Precision: 0.7949
Recall: 0.7798
F1-score: 0.7775
MCC: 0.6779
混淆矩陣:
[[296  48   0]
 [  0 200 156]
 [  0  34 347]]
源域測試集準確率: 0.9643
目標域測試集準確率: 0.7798

實驗完成！所有結果已保存。


In [5]:
# WISDM to UCI baseline
import numpy as np

def main():
    """
    主函數：執行基線模型（不加 MMD）的遷移學習實驗流程
    """
    print("開始執行基線模型實驗（不加 MMD）")
    
    # 創建結果目錄
    results_dir = f"results_baseline_{datetime.now().strftime('%Y%m%d_%H%M%S')}"
    os.makedirs(results_dir, exist_ok=True)
    
    # 加載並處理數據
    print("加載並處理數據...")
    domains = split_and_setup_domains(source='WISDM', target='UCI')  # 調換源域和目標域

    # 對數據進行標準化
    print("對數據進行標準化...")
    domains = normalize_data(domains, method='domain_specific')

    # 提取訓練、測試數據
    source_train_data = domains['source']['train']['data']
    source_train_labels = domains['source']['train']['labels']
    source_test_data = domains['source']['test']['data']
    source_test_labels = domains['source']['test']['labels']

    target_train_data = domains['target']['train']['data']
    target_train_labels = domains['target']['train']['labels']
    target_test_data = domains['target']['test']['data']
    target_test_labels = domains['target']['test']['labels']

    # 分割訓練數據為訓練集和驗證集
    source_train_data, source_val_data, source_train_labels, source_val_labels = train_test_split(
        source_train_data, source_train_labels, test_size=0.2, random_state=42, stratify=source_train_labels
    )
    target_train_data, target_val_data, target_train_labels, target_val_labels = train_test_split(
        target_train_data, target_train_labels, test_size=0.2, random_state=42, stratify=target_train_labels
    )

    print(f"源域數據集: 訓練集 {len(source_train_data)}筆, 驗證集 {len(source_val_data)}筆, 測試集 {len(source_test_data)}筆")
    print(f"目標域數據集: 訓練集 {len(target_train_data)}筆, 驗證集 {len(target_val_data)}筆, 測試集 {len(target_test_data)}筆")
    
    # 超參數
    best_params = {
        'num_rules': 3,          # 規則數量
        'batch_size': 64,        # 批次大小
        'learning_rate': 0.001,  # 學習率
        'mmd_weight': 0.0
    }

    # 訓練多次並儲存結果
    num_repeats = 3  # 設定重複次數
    all_results = []  # 儲存每次訓練的結果
    
    for i in range(num_repeats):
        print(f"\n開始第 {i+1} 次訓練...")
        
        # 創建專屬於本次訓練的結果目錄
        repeat_dir = os.path.join(results_dir, f"repeat_{i+1}")
        os.makedirs(repeat_dir, exist_ok=True)
        
        # 定義 Early Stopping 和 Model Checkpoint
        early_stopping = EarlyStopping(
            monitor='val_accuracy',  # 基於驗證準確率進行早停
            mode='max',
            patience=15,
            verbose=1
        )

        checkpoint_path = os.path.join(repeat_dir, "2LSTM-CTFNN_baseline_wisdm_to_uci_domain_specific.weights.h5")  # 修改檔案名稱
        model_checkpoint = ModelCheckpoint(
            filepath=checkpoint_path,
            monitor='val_accuracy',  # 基於驗證準確率保存最佳模型
            mode='max',
            save_best_only=True,
            verbose=1
        )
        
        # 訓練模型 - 使用驗證集而非測試集進行驗證
        history = train_fuzzy_tsk_mmd_model(
            source_train_data=source_train_data,
            source_train_labels=source_train_labels,
            target_train_data=target_train_data,
            source_val_data=source_val_data,  # 使用驗證集
            source_val_labels=source_val_labels,  # 使用驗證集
            target_val_data=target_val_data,  # 使用驗證集
            target_val_labels=target_val_labels,  # 使用驗證集
            input_shape=source_train_data.shape[1:],  # 輸入數據形狀
            num_classes=source_train_labels.shape[1],  # 類別數
            num_rules=best_params['num_rules'],
            batch_size=best_params['batch_size'],
            learning_rate=best_params['learning_rate'],
            mmd_strategy='static',
            start_mmd_weight=best_params['mmd_weight'],
            epochs=50,
            checkpoint_path=checkpoint_path,  # 傳入模型保存路徑
            patience=15  # 傳入早停的耐心值
        )
        
        # 載入最佳權重
        print("\n載入最佳模型權重...")
        model, _ = create_bilstm_cnn_fuzzy_tsk_model_with_mmd(
            input_shape=source_train_data.shape[1:],
            num_classes=source_train_labels.shape[1],
            num_rules=best_params['num_rules']
        )
        model.load_weights(checkpoint_path)

        # 評估模型 - 使用測試集進行最終評估
        print("\n評估模型性能...")
        source_acc, source_precision, source_recall, source_f1, source_mcc, cm_source = evaluate_model(
            model, source_test_data, source_test_labels, "源域 (WISDM)"  # 修改描述
        )

        target_acc, target_precision, target_recall, target_f1, target_mcc, cm_target = evaluate_model(
            model, target_test_data, target_test_labels, "目標域 (UCI)"  # 修改描述
        )

        print(f"源域測試集準確率: {source_acc:.4f}")
        print(f"目標域測試集準確率: {target_acc:.4f}")

        # 儲存結果
        all_results.append({
            "repeat": i + 1,
            "source_acc": source_acc,
            "source_precision": source_precision,  # 新增精確率
            "source_recall": source_recall,
            "source_f1": source_f1,
            "source_mcc": source_mcc,
            "target_acc": target_acc,
            "target_precision": target_precision,  # 新增精確率
            "target_recall": target_recall,
            "target_f1": target_f1,
            "target_mcc": target_mcc,
            "cm_source": cm_source,
            "cm_target": cm_target
        })

    # 計算平均值和標準差
    metrics_keys = ["source_acc", "source_precision", "source_recall", "source_f1", "source_mcc", 
                    "target_acc", "target_precision", "target_recall", "target_f1", "target_mcc"]
    summary_stats = {key: {"mean": None, "std": None} for key in metrics_keys}

    for key in metrics_keys:
        values = [result[key] for result in all_results]
        summary_stats[key]["mean"] = np.mean(values)
        summary_stats[key]["std"] = np.std(values)

    # 將所有結果保存到總結文件
    summary_file = os.path.join(results_dir, "summary_results_domain_specific.txt")
    with open(summary_file, "w") as f:
        for result in all_results:
            f.write(f"Repeat {result['repeat']}:\n")
            f.write(f"源域準確率: {result['source_acc']:.4f}, Precision: {result['source_precision']:.4f}, "
                    f"Recall: {result['source_recall']:.4f}, F1-score: {result['source_f1']:.4f}, MCC: {result['source_mcc']:.4f}\n")
            f.write(f"目標域準確率: {result['target_acc']:.4f}, Precision: {result['target_precision']:.4f}, "
                    f"Recall: {result['target_recall']:.4f}, F1-score: {result['target_f1']:.4f}, MCC: {result['target_mcc']:.4f}\n")
            f.write("\n")
        
        f.write("\n平均值與標準差:\n")
        for key in metrics_keys:
            f.write(f"{key}: 平均值 = {summary_stats[key]['mean']:.4f} ± {summary_stats[key]['std']:.4f}\n")

    print("\n實驗完成！所有結果已保存。")
    
if __name__ == "__main__":
    main()


開始執行基線模型實驗（不加 MMD）
加載並處理數據...
UCI 數據集類別分佈:
  類別 0: 1722 樣本 (31.86%)
  類別 1: 1777 樣本 (32.88%)
  類別 2: 1906 樣本 (35.26%)

WISDM 數據集類別分佈:
  類別 0: 2308 樣本 (30.55%)
  類別 1: 2742 樣本 (36.29%)
  類別 2: 2506 樣本 (33.17%)

源域 (WISDM) 訓練數據形狀: (6044, 128, 6), 標籤形狀: (6044, 3)
源域 (WISDM) 測試數據形狀: (1512, 128, 6), 標籤形狀: (1512, 3)
目標域 (UCI) 訓練數據形狀: (4324, 128, 6), 標籤形狀: (4324, 3)
目標域 (UCI) 測試數據形狀: (1081, 128, 6), 標籤形狀: (1081, 3)
對數據進行標準化...

標準化完成，方法: domain_specific
源域數據集: 訓練集 4835筆, 驗證集 1209筆, 測試集 1512筆
目標域數據集: 訓練集 3459筆, 驗證集 865筆, 測試集 1081筆

開始第 1 次訓練...

模型架構:
Model: "model_12"
__________________________________________________________________________________________________
Layer (type)                    Output Shape         Param #     Connected to                     
input_7 (InputLayer)            [(None, 128, 6)]     0                                            
__________________________________________________________________________________________________
bidirectional_12 (Bidirectional (Non

Epoch 1/50: 100%|██████████████████████████████████████████████████████████████████████| 76/76 [00:12<00:00,  6.07it/s]


儲存最佳權重，混合指標 = 0.6577, 目標域驗證準確率 = 0.5827
Loss: 0.6539, Class Loss: 0.6539, MMD Loss: 0.2209, Acc: 0.8396
Val Source Acc: 0.9065, Val Target Acc: 0.5827
Mixed Metric: 0.6577, MMD Weight: 0.0000, Time: 14.03s

Epoch 2/50


Epoch 2/50: 100%|██████████████████████████████████████████████████████████████████████| 76/76 [00:12<00:00,  6.15it/s]


儲存最佳權重，混合指標 = 0.7057, 目標域驗證準確率 = 0.6451
Loss: 0.4142, Class Loss: 0.4142, MMD Loss: 0.2432, Acc: 0.9475
Val Source Acc: 0.9280, Val Target Acc: 0.6451
Mixed Metric: 0.7057, MMD Weight: 0.0000, Time: 13.52s

Epoch 3/50


...

Loss: 0.0660, Class Loss: 0.0660, MMD Loss: 0.2722, Acc: 0.9936
Val Source Acc: 0.9950, Val Target Acc: 0.4902
Mixed Metric: 0.6144, MMD Weight: 0.0000, Time: 12.97s

Epoch 25/50


Epoch 25/50: 100%|█████████████████████████████████████████████████████████████████████| 76/76 [00:12<00:00,  6.17it/s]


Loss: 0.0503, Class Loss: 0.0503, MMD Loss: 0.2807, Acc: 0.9981
Val Source Acc: 0.9934, Val Target Acc: 0.5538
Mixed Metric: 0.6576, MMD Weight: 0.0000, Time: 13.05s

Epoch 26/50


Epoch 26/50: 100%|█████████████████████████████████████████████████████████████████████| 76/76 [00:12<00:00,  6.23it/s]


早停於第 26 個 epoch，最佳混合指標 = 0.8137, 最佳目標域驗證準確率 = 0.7838
加載最佳權重完成

載入最佳模型權重...

評估模型性能...
 評估結果:
準確率: 0.9854
Precision: 0.9856
Recall: 0.9854
F1-score: 0.9854
MCC: 0.9782
混淆矩陣:
[[461   1   0]
 [  1 544   4]
 [  1  15 485]]
 評估結果:
準確率: 0.7743
Precision: 0.7855
Recall: 0.7743
F1-score: 0.7782
MCC: 0.6619
混淆矩陣:
[[310   0  34]
 [  0 245 111]
 [  0  99 282]]
源域測試集準確率: 0.9854
目標域測試集準確率: 0.7743

實驗完成！所有結果已保存。


In [6]:
# UCI to WISDM MMD
import os
import numpy as np
from datetime import datetime
from sklearn.model_selection import train_test_split
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint

def main():
    """
    主函數：執行整個遷移學習實驗流程
    """
    print("開始執行遷移學習實驗")
    
    # 創建結果目錄
    results_dir = f"results_{datetime.now().strftime('%Y%m%d_%H%M%S')}"
    os.makedirs(results_dir, exist_ok=True)
    
    # 加載並處理數據
    print("加載並處理數據...")
    domains = split_and_setup_domains(source='UCI', target='WISDM')

    # 對數據進行標準化
    print("對數據進行標準化...")
    domains = normalize_data(domains, method='domain_specific')

    # 提取訓練、測試數據
    source_train_data = domains['source']['train']['data']
    source_train_labels = domains['source']['train']['labels']
    source_test_data = domains['source']['test']['data']
    source_test_labels = domains['source']['test']['labels']

    target_train_data = domains['target']['train']['data']
    target_train_labels = domains['target']['train']['labels']
    target_test_data = domains['target']['test']['data']
    target_test_labels = domains['target']['test']['labels']

    # 分割訓練數據為訓練集和驗證集
    source_train_data, source_val_data, source_train_labels, source_val_labels = train_test_split(
        source_train_data, source_train_labels, test_size=0.2, random_state=42, stratify=source_train_labels
    )
    target_train_data, target_val_data, target_train_labels, target_val_labels = train_test_split(
        target_train_data, target_train_labels, test_size=0.2, random_state=42, stratify=target_train_labels
    )

    print(f"源域數據集: 訓練集 {len(source_train_data)}筆, 驗證集 {len(source_val_data)}筆, 測試集 {len(source_test_data)}筆")
    print(f"目標域數據集: 訓練集 {len(target_train_data)}筆, 驗證集 {len(target_val_data)}筆, 測試集 {len(target_test_data)}筆")
    
    # 超參數
    best_params = {
        'num_rules': 3,          # 規則數量
        'batch_size': 64,        # 批次大小
        'learning_rate': 0.001,  # 學習率
        'mmd_weight': 0.5       # MMD 權重
    }

    # 訓練多次並儲存結果
    num_repeats = 3  # 設定重複次數
    all_results = []  # 儲存每次訓練的結果
    
    for i in range(num_repeats):
        print(f"\n開始第 {i+1} 次訓練...")
        
        # 創建專屬於本次訓練的結果目錄
        repeat_dir = os.path.join(results_dir, f"repeat_{i+1}")
        os.makedirs(repeat_dir, exist_ok=True)
        
        # 定義 Early Stopping 和 Model Checkpoint
        early_stopping = EarlyStopping(
            monitor='val_target_accuracy',  # 修改為有效指標名稱
            mode='max',
            patience=15,
            verbose=1
        )

        checkpoint_path = os.path.join(repeat_dir, "2LSTM-CTFNN_uci_to_wisdm_MMD_domain_specificMMD05.weights.h5")
        model_checkpoint = ModelCheckpoint(
            filepath=checkpoint_path,
            monitor='val_target_accuracy',  # 修改為有效指標名稱
            mode='max',
            save_best_only=True,
            verbose=1
        )
        
        # 訓練模型 - 使用驗證集而非測試集進行驗證
        history = train_fuzzy_tsk_mmd_model(
            source_train_data=source_train_data,
            source_train_labels=source_train_labels,
            target_train_data=target_train_data,
            source_val_data=source_val_data,  # 使用驗證集
            source_val_labels=source_val_labels,  # 使用驗證集
            target_val_data=target_val_data,  # 使用驗證集
            target_val_labels=target_val_labels,  # 使用驗證集
            input_shape=source_train_data.shape[1:],  # 輸入數據形狀
            num_classes=source_train_labels.shape[1],  # 類別數
            num_rules=best_params['num_rules'],
            batch_size=best_params['batch_size'],
            learning_rate=best_params['learning_rate'],
            mmd_strategy='static',
            start_mmd_weight=best_params['mmd_weight'],
            epochs=50,
            checkpoint_path=checkpoint_path,  # 傳入模型保存路徑
            patience=15  # 傳入早停的耐心值
        )
        
        # 載入最佳權重
        print("\n載入最佳模型權重...")
        model, _ = create_bilstm_cnn_fuzzy_tsk_model_with_mmd(
            input_shape=source_train_data.shape[1:],
            num_classes=source_train_labels.shape[1],
            num_rules=best_params['num_rules']
        )
        model.load_weights(checkpoint_path)

        # 評估模型 - 使用測試集進行最終評估
        print("\n評估模型性能...")
        source_acc, source_precision, source_recall, source_f1, source_mcc, cm_source = evaluate_model(
            model, source_test_data, source_test_labels, None, "源域 (UCI)", save_dir=repeat_dir
        )

        target_acc, target_precision, target_recall, target_f1, target_mcc, cm_target = evaluate_model(
            model, target_test_data, target_test_labels, None, "目標域 (WISDM)", save_dir=repeat_dir
        )

        print(f"源域測試集準確率: {source_acc:.4f}")
        print(f"目標域測試集準確率: {target_acc:.4f}")

        # 儲存每次訓練的原域和目標域指標到單獨文件
        with open(os.path.join(repeat_dir, "source_metrics.txt"), "w") as f:
            f.write(f"準確率: {source_acc:.4f}\n")
            f.write(f"Precision: {source_precision:.4f}\n")
            f.write(f"Recall: {source_recall:.4f}\n")
            f.write(f"F1-score: {source_f1:.4f}\n")
            f.write(f"MCC: {source_mcc:.4f}\n")
        
        with open(os.path.join(repeat_dir, "target_metrics.txt"), "w") as f:
            f.write(f"準確率: {target_acc:.4f}\n")
            f.write(f"Precision: {target_precision:.4f}\n")
            f.write(f"Recall: {target_recall:.4f}\n")
            f.write(f"F1-score: {target_f1:.4f}\n")
            f.write(f"MCC: {target_mcc:.4f}\n")

        # 儲存結果
        all_results.append({
            "repeat": i + 1,
            "source_acc": source_acc,
            "source_precision": source_precision,
            "source_recall": source_recall,
            "source_f1": source_f1,
            "source_mcc": source_mcc,
            "target_acc": target_acc,
            "target_precision": target_precision,
            "target_recall": target_recall,
            "target_f1": target_f1,
            "target_mcc": target_mcc,
            "cm_source": cm_source,
            "cm_target": cm_target
        })

    # 計算平均值和標準差
    metrics_keys = ["source_acc", "source_precision", "source_recall", "source_f1", "source_mcc", 
                    "target_acc", "target_precision", "target_recall", "target_f1", "target_mcc"]
    summary_stats = {key: {"mean": None, "std": None} for key in metrics_keys}

    for key in metrics_keys:
        values = [result[key] for result in all_results]
        summary_stats[key]["mean"] = np.mean(values)
        summary_stats[key]["std"] = np.std(values)

    # 將所有結果保存到總結文件
    summary_file = os.path.join(results_dir, "summary_results_domain_specificMMD05.txt")
    with open(summary_file, "w") as f:
        for result in all_results:
            f.write(f"Repeat {result['repeat']}:\n")
            f.write(f"源域準確率: {result['source_acc']:.4f}, Precision: {result['source_precision']:.4f}, "
                    f"Recall: {result['source_recall']:.4f}, F1-score: {result['source_f1']:.4f}, MCC: {result['source_mcc']:.4f}\n")
            f.write(f"目標域準確率: {result['target_acc']:.4f}, Precision: {result['target_precision']:.4f}, "
                    f"Recall: {result['target_recall']:.4f}, F1-score: {result['target_f1']:.4f}, MCC: {result['target_mcc']:.4f}\n")
            f.write("\n")
        
        f.write("\n平均值與標準差:\n")
        for key in metrics_keys:
            f.write(f"{key}: 平均值 = {summary_stats[key]['mean']:.4f} ± {summary_stats[key]['std']:.4f}\n")

    print("\n實驗完成！所有結果已保存。")
    
if __name__ == "__main__":
    main()


開始執行遷移學習實驗
加載並處理數據...
UCI 數據集類別分佈:
  類別 0: 1722 樣本 (31.86%)
  類別 1: 1777 樣本 (32.88%)
  類別 2: 1906 樣本 (35.26%)

WISDM 數據集類別分佈:
  類別 0: 2308 樣本 (30.55%)
  類別 1: 2742 樣本 (36.29%)
  類別 2: 2506 樣本 (33.17%)

源域 (UCI) 訓練數據形狀: (4324, 128, 6), 標籤形狀: (4324, 3)
源域 (UCI) 測試數據形狀: (1081, 128, 6), 標籤形狀: (1081, 3)
目標域 (WISDM) 訓練數據形狀: (6044, 128, 6), 標籤形狀: (6044, 3)
目標域 (WISDM) 測試數據形狀: (1512, 128, 6), 標籤形狀: (1512, 3)
對數據進行標準化...

標準化完成，方法: domain_specific
源域數據集: 訓練集 3459筆, 驗證集 865筆, 測試集 1081筆
目標域數據集: 訓練集 4835筆, 驗證集 1209筆, 測試集 1512筆

開始第 1 次訓練...

模型架構:
Model: "model_24"
__________________________________________________________________________________________________
Layer (type)                    Output Shape         Param #     Connected to                     
input_13 (InputLayer)           [(None, 128, 6)]     0                                            
__________________________________________________________________________________________________
bidirectional_24 (Bidirectional (None, 128, 

Epoch 1/50: 100%|██████████████████████████████████████████████████████████████████████| 55/55 [00:09<00:00,  5.75it/s]


儲存最佳權重，混合指標 = 0.4775, 目標域驗證準確率 = 0.4624
Loss: 1.0037, Class Loss: 0.9212, MMD Loss: 0.1649, Acc: 0.5198
Val Source Acc: 0.5676, Val Target Acc: 0.4624
Mixed Metric: 0.4775, MMD Weight: 0.5000, Time: 11.11s

Epoch 2/50


Epoch 2/50: 100%|██████████████████████████████████████████████████████████████████████| 55/55 [00:08<00:00,  6.27it/s]


Loss: 0.8424, Class Loss: 0.7661, MMD Loss: 0.1526, Acc: 0.5935
Val Source Acc: 0.5549, Val Target Acc: 0.3507
Mixed Metric: 0.3967, MMD Weight: 0.5000, Time: 9.52s

Epoch 3/50


...

Loss: 0.1260, Class Loss: 0.0645, MMD Loss: 0.1231, Acc: 0.9767
Val Source Acc: 0.9595, Val Target Acc: 0.6038
Mixed Metric: 0.6982, MMD Weight: 0.5000, Time: 9.67s

Epoch 30/50


Epoch 30/50: 100%|█████████████████████████████████████████████████████████████████████| 55/55 [00:08<00:00,  6.13it/s]


Loss: 0.1319, Class Loss: 0.0675, MMD Loss: 0.1288, Acc: 0.9764
Val Source Acc: 0.9618, Val Target Acc: 0.5401
Mixed Metric: 0.6538, MMD Weight: 0.5000, Time: 9.67s

Epoch 31/50


Epoch 31/50: 100%|█████████████████████████████████████████████████████████████████████| 55/55 [00:08<00:00,  6.18it/s]


早停於第 31 個 epoch，最佳混合指標 = 0.7558, 最佳目標域驗證準確率 = 0.6989
加載最佳權重完成

載入最佳模型權重...

評估模型性能...
源域 (UCI) 評估結果:
準確率: 0.9454
Precision: 0.9456
Recall: 0.9454
F1-score: 0.9454
MCC: 0.9182
混淆矩陣:
[[344   0   0]
 [  0 331  25]
 [  0  34 347]]
混淆矩陣已保存至: results_20260703_114134\repeat_3\源域 (UCI)_confusion_matrix.csv
目標域 (WISDM) 評估結果:
準確率: 0.6931
Precision: 0.7142
Recall: 0.6931
F1-score: 0.7004
MCC: 0.5386
混淆矩陣:
[[377  47  38]
 [  1 365 183]
 [  3 192 306]]
混淆矩陣已保存至: results_20260703_114134\repeat_3\目標域 (WISDM)_confusion_matrix.csv
源域測試集準確率: 0.9454
目標域測試集準確率: 0.6931

實驗完成！所有結果已保存。


In [7]:
# WISDM to UCI MMD
import os
import numpy as np
from datetime import datetime
from sklearn.model_selection import train_test_split
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint

def main():
    """
    主函數：執行整個遷移學習實驗流程
    """
    print("開始執行遷移學習實驗")
    
    # 創建結果目錄
    results_dir = f"results_{datetime.now().strftime('%Y%m%d_%H%M%S')}"
    os.makedirs(results_dir, exist_ok=True)
    
    # 加載並處理數據
    print("加載並處理數據...")
    domains = split_and_setup_domains(source='WISDM', target='UCI')  # 將源域設為 WISDM，目標域設為 UCI

    # 對數據進行標準化
    print("對數據進行標準化...")
    domains = normalize_data(domains, method='domain_specific')

    # 提取訓練、測試數據
    source_train_data = domains['source']['train']['data']
    source_train_labels = domains['source']['train']['labels']
    source_test_data = domains['source']['test']['data']
    source_test_labels = domains['source']['test']['labels']

    target_train_data = domains['target']['train']['data']
    target_train_labels = domains['target']['train']['labels']
    target_test_data = domains['target']['test']['data']
    target_test_labels = domains['target']['test']['labels']

    # 分割訓練數據為訓練集和驗證集
    source_train_data, source_val_data, source_train_labels, source_val_labels = train_test_split(
        source_train_data, source_train_labels, test_size=0.2, random_state=42, stratify=source_train_labels
    )
    target_train_data, target_val_data, target_train_labels, target_val_labels = train_test_split(
        target_train_data, target_train_labels, test_size=0.2, random_state=42, stratify=target_train_labels
    )

    print(f"源域數據集: 訓練集 {len(source_train_data)}筆, 驗證集 {len(source_val_data)}筆, 測試集 {len(source_test_data)}筆")
    print(f"目標域數據集: 訓練集 {len(target_train_data)}筆, 驗證集 {len(target_val_data)}筆, 測試集 {len(target_test_data)}筆")
    
    # 超參數
    best_params = {
        'num_rules': 3,          # 規則數量
        'batch_size': 64,        # 批次大小
        'learning_rate': 0.001,  # 學習率
        'mmd_weight': 0.5        # MMD 權重
    }

    # 訓練多次並儲存結果
    num_repeats = 3  # 設定重複次數
    all_results = []  # 儲存每次訓練的結果
    
    for i in range(num_repeats):
        print(f"\n開始第 {i+1} 次訓練...")
        
        # 創建專屬於本次訓練的結果目錄
        repeat_dir = os.path.join(results_dir, f"repeat_{i+1}")
        os.makedirs(repeat_dir, exist_ok=True)
        
        # 定義 Early Stopping 和 Model Checkpoint
        early_stopping = EarlyStopping(
            monitor='val_target_accuracy',  # 修改為有效指標名稱
            mode='max',
            patience=15,
            verbose=1
        )

        checkpoint_path = os.path.join(repeat_dir, "2LSTM-CTFNN_wisdm_to_uci_MMD_domain_specific_MMD05.weights.h5")  # 修改檔案名稱
        model_checkpoint = ModelCheckpoint(
            filepath=checkpoint_path,
            monitor='val_target_accuracy',  # 修改為有效指標名稱
            mode='max',
            save_best_only=True,
            verbose=1
        )
        
        # 訓練模型 - 使用驗證集而非測試集進行驗證
        history = train_fuzzy_tsk_mmd_model(
            source_train_data=source_train_data,
            source_train_labels=source_train_labels,
            target_train_data=target_train_data,
            source_val_data=source_val_data,  # 使用驗證集
            source_val_labels=source_val_labels,  # 使用驗證集
            target_val_data=target_val_data,  # 使用驗證集
            target_val_labels=target_val_labels,  # 使用驗證集
            input_shape=source_train_data.shape[1:],  # 輸入數據形狀
            num_classes=source_train_labels.shape[1],  # 類別數
            num_rules=best_params['num_rules'],
            batch_size=best_params['batch_size'],
            learning_rate=best_params['learning_rate'],
            mmd_strategy='static',
            start_mmd_weight=best_params['mmd_weight'],
            epochs=50,
            checkpoint_path=checkpoint_path,  # 傳入模型保存路徑
            patience=15  # 傳入早停的耐心值
        )
        
        # 載入最佳權重
        print("\n載入最佳模型權重...")
        model, _ = create_bilstm_cnn_fuzzy_tsk_model_with_mmd(
            input_shape=source_train_data.shape[1:],
            num_classes=source_train_labels.shape[1],
            num_rules=best_params['num_rules']
        )
        model.load_weights(checkpoint_path)

        # 評估模型 - 使用測試集進行最終評估
        print("\n評估模型性能...")
        source_acc, source_precision, source_recall, source_f1, source_mcc, cm_source = evaluate_model(
            model, source_test_data, source_test_labels, "源域 (WISDM)"
        )

        target_acc, target_precision, target_recall, target_f1, target_mcc, cm_target = evaluate_model(
            model, target_test_data, target_test_labels, "目標域 (UCI)"
        )

        print(f"源域測試集準確率: {source_acc:.4f}")
        print(f"目標域測試集準確率: {target_acc:.4f}")

        # 儲存每次訓練的原域和目標域指標到單獨文件
        with open(os.path.join(repeat_dir, "source_metrics.txt"), "w") as f:
            f.write(f"準確率: {source_acc:.4f}\n")
            f.write(f"Precision: {source_precision:.4f}\n")
            f.write(f"Recall: {source_recall:.4f}\n")
            f.write(f"F1-score: {source_f1:.4f}\n")
            f.write(f"MCC: {source_mcc:.4f}\n")
        
        with open(os.path.join(repeat_dir, "target_metrics.txt"), "w") as f:
            f.write(f"準確率: {target_acc:.4f}\n")
            f.write(f"Precision: {target_precision:.4f}\n")
            f.write(f"Recall: {target_recall:.4f}\n")
            f.write(f"F1-score: {target_f1:.4f}\n")
            f.write(f"MCC: {target_mcc:.4f}\n")

        # 儲存混淆矩陣到文件
        np.savetxt(os.path.join(repeat_dir, "source_confusion_matrix.txt"), cm_source, fmt='%d')
        np.savetxt(os.path.join(repeat_dir, "target_confusion_matrix.txt"), cm_target, fmt='%d')

        # 儲存結果
        all_results.append({
            "repeat": i + 1,
            "source_acc": source_acc,
            "source_precision": source_precision,
            "source_recall": source_recall,
            "source_f1": source_f1,
            "source_mcc": source_mcc,
            "target_acc": target_acc,
            "target_precision": target_precision,
            "target_recall": target_recall,
            "target_f1": target_f1,
            "target_mcc": target_mcc,
            "cm_source": cm_source,
            "cm_target": cm_target
        })

    # 計算平均值和標準差
    metrics_keys = ["source_acc", "source_precision", "source_recall", "source_f1", "source_mcc", 
                    "target_acc", "target_precision", "target_recall", "target_f1", "target_mcc"]
    summary_stats = {key: {"mean": None, "std": None} for key in metrics_keys}

    for key in metrics_keys:
        values = [result[key] for result in all_results]
        summary_stats[key]["mean"] = np.mean(values)
        summary_stats[key]["std"] = np.std(values)

    # 將所有結果保存到總結文件
    summary_file = os.path.join(results_dir, "summary_results_domain_specific_MMD05.txt")
    with open(summary_file, "w") as f:
        for result in all_results:
            f.write(f"Repeat {result['repeat']}:\n")
            f.write(f"源域準確率: {result['source_acc']:.4f}, Precision: {result['source_precision']:.4f}, "
                    f"Recall: {result['source_recall']:.4f}, F1-score: {result['source_f1']:.4f}, MCC: {result['source_mcc']:.4f}\n")
            f.write(f"目標域準確率: {result['target_acc']:.4f}, Precision: {result['target_precision']:.4f}, "
                    f"Recall: {result['target_recall']:.4f}, F1-score: {result['target_f1']:.4f}, MCC: {result['target_mcc']:.4f}\n")
            f.write("\n")
        
        f.write("\n平均值與標準差:\n")
        for key in metrics_keys:
            f.write(f"{key}: 平均值 = {summary_stats[key]['mean']:.4f} ± {summary_stats[key]['std']:.4f}\n")

    print("\n實驗完成！所有結果已保存。")
    
if __name__ == "__main__":
    main()


開始執行遷移學習實驗
加載並處理數據...
UCI 數據集類別分佈:
  類別 0: 1722 樣本 (31.86%)
  類別 1: 1777 樣本 (32.88%)
  類別 2: 1906 樣本 (35.26%)

WISDM 數據集類別分佈:
  類別 0: 2308 樣本 (30.55%)
  類別 1: 2742 樣本 (36.29%)
  類別 2: 2506 樣本 (33.17%)

源域 (WISDM) 訓練數據形狀: (6044, 128, 6), 標籤形狀: (6044, 3)
源域 (WISDM) 測試數據形狀: (1512, 128, 6), 標籤形狀: (1512, 3)
目標域 (UCI) 訓練數據形狀: (4324, 128, 6), 標籤形狀: (4324, 3)
目標域 (UCI) 測試數據形狀: (1081, 128, 6), 標籤形狀: (1081, 3)
對數據進行標準化...

標準化完成，方法: domain_specific
源域數據集: 訓練集 4835筆, 驗證集 1209筆, 測試集 1512筆
目標域數據集: 訓練集 3459筆, 驗證集 865筆, 測試集 1081筆

開始第 1 次訓練...

模型架構:
Model: "model_36"
__________________________________________________________________________________________________
Layer (type)                    Output Shape         Param #     Connected to                     
input_19 (InputLayer)           [(None, 128, 6)]     0                                            
__________________________________________________________________________________________________
bidirectional_36 (Bidirectional (None, 128, 

Epoch 1/50: 100%|██████████████████████████████████████████████████████████████████████| 76/76 [00:12<00:00,  6.07it/s]


儲存最佳權重，混合指標 = 0.4712, 目標域驗證準確率 = 0.4243
Loss: 0.9375, Class Loss: 0.8745, MMD Loss: 0.1260, Acc: 0.6073
Val Source Acc: 0.6228, Val Target Acc: 0.4243
Mixed Metric: 0.4712, MMD Weight: 0.5000, Time: 14.06s

Epoch 2/50


Epoch 2/50: 100%|██████████████████████████████████████████████████████████████████████| 76/76 [00:12<00:00,  6.20it/s]


儲存最佳權重，混合指標 = 0.7523, 目標域驗證準確率 = 0.7017
Loss: 0.7244, Class Loss: 0.6624, MMD Loss: 0.1240, Acc: 0.6520
Val Source Acc: 0.9115, Val Target Acc: 0.7017
Mixed Metric: 0.7523, MMD Weight: 0.5000, Time: 13.17s

Epoch 3/50


...

Loss: 0.0626, Class Loss: 0.0138, MMD Loss: 0.0976, Acc: 0.9967
Val Source Acc: 0.9876, Val Target Acc: 0.7965
Mixed Metric: 0.8441, MMD Weight: 0.5000, Time: 13.06s

Epoch 22/50


Epoch 22/50: 100%|█████████████████████████████████████████████████████████████████████| 76/76 [00:12<00:00,  6.20it/s]


Loss: 0.1048, Class Loss: 0.0535, MMD Loss: 0.1026, Acc: 0.9822
Val Source Acc: 0.9826, Val Target Acc: 0.7896
Mixed Metric: 0.8372, MMD Weight: 0.5000, Time: 12.98s

Epoch 23/50


Epoch 23/50: 100%|█████████████████████████████████████████████████████████████████████| 76/76 [00:12<00:00,  6.13it/s]


早停於第 23 個 epoch，最佳混合指標 = 0.8503, 最佳目標域驗證準確率 = 0.8116
加載最佳權重完成

載入最佳模型權重...

評估模型性能...
 評估結果:
準確率: 0.9742
Precision: 0.9742
Recall: 0.9742
F1-score: 0.9742
MCC: 0.9612
混淆矩陣:
[[462   0   0]
 [  1 529  19]
 [  4  15 482]]
 評估結果:
準確率: 0.8363
Precision: 0.8357
Recall: 0.8363
F1-score: 0.8360
MCC: 0.7542
混淆矩陣:
[[342   1   1]
 [  2 269  85]
 [  3  85 293]]
源域測試集準確率: 0.9742
目標域測試集準確率: 0.8363

實驗完成！所有結果已保存。
